# HemoDyn-AI v2: Computational Fluid Dynamics (CFD) Progression Benchmark
**Author**: Jace E. Roman (Temple University)  
**Contact**: jace.roman@temple.edu  
**License**: MIT

---

### Overview
This notebook walks through the golden-standard 3D Navier-Stokes finite element simulations computed across **4 arterial stenosis configurations** using SimVascular (`svSolver`):
- **Healthy (0%)**: Baseline idealized straight cylindrical vessel ($D = 4.0\text{ mm}$, $L = 120\text{ mm}$)
- **Mild Stenosis (30%)**: Cosine lumen reduction ($D_{\text{throat}} = 3.34\text{ mm}$)
- **Moderate Stenosis (50%)**: Intermediate lumen constriction ($D_{\text{throat}} = 2.82\text{ mm}$)
- **Severe Stenosis (70%)**: Critical narrowing ($D_{\text{throat}} = 2.19\text{ mm}$)

#### Governing Navier-Stokes Equations
$$\rho \left( \frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} \right) = -\nabla p + \mu \nabla^2 \mathbf{u}$$
$$\nabla \cdot \mathbf{u} = 0$$

Fluid properties:
- Dynamic viscosity: $\mu = 0.004\text{ Pa}\cdot\text{s}$ ($0.04\text{ P}$)
- Density: $\rho = 1060\text{ kg/m}^3$ ($1.06\text{ g/cm}^3$)
- Inflow flow rate: $Q = 1244.07\text{ mm}^3/\text{s}$ ($1.244\text{ ml/s}$)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(str(Path("..").resolve() / "src"))
from hemodyn.physics import calculate_poiseuille_velocity, compute_reynolds_number
from hemodyn.plotting import set_publication_style

set_publication_style()
print("HemoDyn-AI v2 environment initialized successfully.")

### 1. 4-Case Hemodynamic Progression Summary
We inspect the converged finite element simulation summary table comparing peak velocities, translesional pressure gradients, and maximum wall shear stress (WSS).

In [ ]:
summary_df = pd.read_csv("../results/metrics/hemodyn_four_case_summary.csv")
summary_df

### 2. Plotting Key Hemodynamic Biomarkers
We plot the progression of **Peak Jet Velocity**, **Translesional Pressure Drop** ($\Delta P$), and **Maximum Wall Shear Stress** (WSS) across stenosis severities.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

cases = summary_df["case"]
x = np.arange(len(cases))

# 1. Peak velocity
axes[0].bar(x, summary_df["peak_velocity_mm_s"], color="#1f77b4", alpha=0.85)
axes[0].set_ylabel("Peak Velocity (mm/s)")
axes[0].set_title("Peak Jet Velocity")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cases)

# 2. Pressure Drop
axes[1].bar(x, summary_df["pressure_drop_cgs_ba"], color="#d62728", alpha=0.85)
axes[1].set_ylabel("Pressure Drop (Barye / dyn/cm²)")
axes[1].set_title("Translesional Pressure Drop")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cases)

# 3. Maximum WSS
axes[2].bar(x, summary_df["max_wss_dyn_cm2"], color="#2ca02c", alpha=0.85)
axes[2].set_ylabel("Max WSS (dyn/cm²)")
axes[2].set_title("Peak Wall Shear Stress")
axes[2].set_xticks(x)
axes[2].set_xticklabels(cases)

plt.tight_layout()
plt.show()

### 3. Anatomical Lumen Diameter Profiles
We visualize the cosine stenosis geometry formulation along the axial length $z \in [0, 120]\text{ mm}$.

In [ ]:
z = np.linspace(0, 120, 500)
d0 = 4.0

fig, ax = plt.subplots(figsize=(10, 4))
for severity, label, col in [(0.0, "Healthy (0%)", "#2ca02c"),
                              (0.30, "Mild (30%)", "#1f77b4"),
                              (0.50, "Moderate (50%)", "#ff7f0e"),
                              (0.70, "Severe (70%)", "#d62728")]:
    stenosis_mask = (z >= 50) & (z <= 70)
    diam = np.full_like(z, d0)
    # Cosine reduction: d(z) = d0 * (1 - S * (1 + cos(pi * (z - 60) / 10)) / 2)
    diam[stenosis_mask] = d0 * (1.0 - (1.0 - np.sqrt(1.0 - severity)) * 0.5 * (1.0 + np.cos(np.pi * (z[stenosis_mask] - 60.0) / 10.0)))
    ax.plot(z, diam, label=label, color=col, linewidth=2)

ax.axvspan(55, 65, color="gray", alpha=0.15, label="Stenosis Throat Region")
ax.set_xlabel("Axial Coordinate z (mm)")
ax.set_ylabel("Lumen Diameter (mm)")
ax.set_title("Longitudinal Arterial Diameter Profiles", fontweight="bold")
ax.legend(loc="lower left")
plt.show()